# 1. Data Acquisition
Duyệt danh sách bệnh nhân trên PhysioNet, chọn cohort, đọc numerics record.

## 1.0 Import & cấu hình

In [ ]:
import wfdb
import pandas as pd
import numpy as np
import requests
import random
import os

DB_NAME = "mimic3wdb-matched"
DB_VERSION = "1.0"
RECORDS_URL = f"https://physionet.org/files/{DB_NAME}/{DB_VERSION}/RECORDS-numerics"

random.seed(42)  

## 1.1 Đọc kết quả sàng lọc từ screening.py
Không chạy sàng lọc trong notebook (đã chuyển sang `screening.py` chạy độc lập, xem lý do ở README/mục trước).

Ở đây chỉ đọc lại `screening_checkpoint.parquet` — kết quả các ứng viên đã pass bộ lọc SPO2/PULSE tồn tại + đủ window.

In [ ]:
DATA_DIR = os.environ.get("DATA_DIR", ".")
screening_df = pd.read_parquet(f"{DATA_DIR}/screening_checkpoint.parquet")
print(f"Đã đọc {len(screening_df)} ứng viên hợp lệ từ checkpoint")
screening_df.head()

## 1.2 Lọc chất lượng + chọn cohort cân bằng theo mức độ nặng
Lọc bỏ ứng viên có `dropout_ratio_spo2` (đo trong 1 cửa sổ ngắn lúc sàng lọc) quá cao, sau đó lấy mẫu theo 4 nhóm mức độ nặng (`severity_bucket` dựa trên `spo2_min_proxy`) theo tỷ lệ mục tiêu — tránh cohort bị lệch tự nhiên về phía SpO2 cao (đa số ICU record là "bình thường phần lớn thời gian"), giữ đủ đại diện các ca nặng cho bước gán nhãn NEWS2 ở Bước 4.

In [ ]:
MAX_DROPOUT_RATIO = 0.20
FINAL_COHORT_SIZE = 850

quality_filtered_df = screening_df[screening_df["dropout_ratio_spo2"] <= MAX_DROPOUT_RATIO].copy()
print(f"Còn lại sau lọc dropout: {len(quality_filtered_df)} / {len(screening_df)}")

# Chia bucket theo mức độ nặng
bins = [0, 90, 94, 97, 101]
labels = ["<90", "90-94", "94-97", ">97"]
quality_filtered_df["severity_bucket"] = pd.cut(
    quality_filtered_df["spo2_min_proxy"], bins=bins, labels=labels, right=False
)
print(quality_filtered_df["severity_bucket"].value_counts())

# Tỷ lệ mong muốn mỗi bucket + thứ tự ưu tiên bù (liền kề mức độ nặng trước)
bucket_quota = {"<90": 0.25, "90-94": 0.30, "94-97": 0.30, ">97": 0.15}
fallback_order = {
    "<90": ["90-94", "94-97", ">97"],
    "90-94": ["<90", "94-97", ">97"],
    "94-97": ["90-94", ">97", "<90"],
    ">97": ["94-97", "90-94", "<90"],
}

target_n = {b: round(FINAL_COHORT_SIZE * frac) for b, frac in bucket_quota.items()}
pools = {
    b: quality_filtered_df[quality_filtered_df["severity_bucket"] == b].sample(frac=1, random_state=42)
    for b in labels
}

sampled_parts = []
deficit = 0
for bucket in labels:
    n_take = min(len(pools[bucket]), target_n[bucket])
    sampled_parts.append(pools[bucket].iloc[:n_take])
    pools[bucket] = pools[bucket].iloc[n_take:]  
    deficit += target_n[bucket] - n_take

# Bù deficit từ các bucket còn dư, theo thứ tự ưu tiên liền kề
if deficit > 0:
    for bucket in labels:
        if deficit <= 0:
            break
        for source in fallback_order[bucket]:
            if deficit <= 0:
                break
            available = pools[source]
            if len(available) == 0:
                continue
            n_borrow = min(len(available), deficit)
            sampled_parts.append(available.iloc[:n_borrow])
            pools[source] = available.iloc[n_borrow:]
            deficit -= n_borrow

cohort_df = pd.concat(sampled_parts).drop_duplicates(subset=["patient_id", "record_path"]).reset_index(drop=True)
print(f"Cohort cuối: {len(cohort_df)} / {FINAL_COHORT_SIZE} mục tiêu")
print(cohort_df["severity_bucket"].value_counts())
if deficit > 0:
    print(f"⚠️ Vẫn thiếu {deficit} bệnh nhân — pool ứng viên chưa đủ, cân nhắc tăng candidate_pool_size.")

### Trực quan hoá severity_bucket trước/sau lấy mẫu
So sánh phân bố mức độ nặng của pool ứng viên (`quality_filtered_df`) với cohort đã lấy mẫu theo quota (`cohort_df`) — xác nhận trực quan việc lấy mẫu có thực sự cân bằng lại tỷ lệ như mục tiêu (25/30/30/15%)
hay không, thay vì chỉ đọc số ở bảng in ra.

In [ ]:
import matplotlib.pyplot as plt

before = quality_filtered_df["severity_bucket"].value_counts().reindex(labels)
after = cohort_df["severity_bucket"].value_counts().reindex(labels)

fig, ax = plt.subplots(figsize=(8, 5))
x = range(len(labels))
ax.bar([i - 0.2 for i in x], before, width=0.4, label="Pool trước lấy mẫu", color="#90CAF9")
ax.bar([i + 0.2 for i in x], after, width=0.4, label="Cohort sau lấy mẫu", color="#F44336")
ax.set_xticks(list(x)); ax.set_xticklabels(labels)
ax.set_ylabel("Số bệnh nhân")
ax.set_title("Phân bố severity_bucket: trước vs sau quota sampling")
ax.legend()
plt.tight_layout()
plt.show()

## 1.3 Đọc numerics record đầy đủ cho cohort đã chọn

In [ ]:
MAX_HOURS_PER_RECORD = 48
TARGET_RESAMPLE = "1min"

def read_full_numerics(patient_id: str, record_path: str) -> pd.DataFrame | None:
    pn_dir = f"{DB_NAME}/{DB_VERSION}/" + "/".join(record_path.split("/")[:-1])
    record_name = record_path.split("/")[-1]

    try:
        header = wfdb.rdheader(record_name, pn_dir=pn_dir)
    except Exception as e:
        print(f"[SKIP-HEADER] {record_path}: {e}")
        return None

    sig_names = [s.upper() for s in header.sig_name]
    if "SPO2" not in sig_names or "PULSE" not in sig_names:
        return None

    max_samples = int(MAX_HOURS_PER_RECORD * 3600 * header.fs)
    sampto = min(header.sig_len, max_samples)

    try:
        rec = wfdb.rdrecord(record_name, pn_dir=pn_dir, sampto=sampto)
    except Exception as e:
        print(f"[SKIP-READ] {record_path}: {e}")
        return None

    df = pd.DataFrame(rec.p_signal, columns=[s.upper() for s in rec.sig_name])
    df = df[["SPO2", "PULSE"]].copy()
    df = df.replace(0, np.nan)

    if rec.base_datetime is not None:
        timestamps = pd.date_range(start=rec.base_datetime, periods=len(df), freq=pd.Timedelta(seconds=1 / rec.fs))
    else:
        timestamps = pd.date_range(start="2000-01-01", periods=len(df), freq=pd.Timedelta(seconds=1 / rec.fs))
    df.index = timestamps

    df_resampled = df.resample(TARGET_RESAMPLE).mean()
    df_resampled = df_resampled.reset_index().rename(columns={"index": "timestamp"})

    df_resampled["patient_id"] = patient_id
    df_resampled["record_path"] = record_path
    df_resampled["source_fs_hz"] = rec.fs
    df_resampled["dropout_ratio_full_record"] = df_resampled["SPO2"].isna().mean()

    return df_resampled

## 1.4 Gom toàn bộ cohort thành một DataFrame duy nhất

In [ ]:
import concurrent.futures
import time
from tqdm.notebook import tqdm

def read_full_numerics_safe(patient_id: str, record_path: str, timeout_sec: int = 20):
    """Wrapper: chạy read_full_numerics với timeout, tránh treo vô thời hạn."""
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
        future = executor.submit(read_full_numerics, patient_id, record_path)
        try:
            return future.result(timeout=timeout_sec)
        except concurrent.futures.TimeoutError:
            print(f"[TIMEOUT >{timeout_sec}s] {patient_id} — {record_path}")
            return None
        except Exception as e:
            print(f"[ERROR] {patient_id}: {e}")
            return None

### Chạy fetch với resume theo patient_id + retry lỗi mạng
Mỗi lần chạy lại cell này chỉ fetch phần
patient_id còn thiếu so với `raw_df.parquet` đã có, và retry backoff cho lỗi mạng (timeout/502) trước khi
tính là fail hẳn — tránh mất % lớn cohort chỉ vì 1 lần mạng chập chờn.

In [ ]:
CHECKPOINT_DIR = "checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
RAW_DF_PATH = f"{CHECKPOINT_DIR}/raw_df.parquet"
FAILED_LOG_PATH = f"{CHECKPOINT_DIR}/raw_df_failed.txt"
MAX_WORKERS = 4

# Resume theo từng patient_id, không phải all-or-nothing
if os.path.exists(RAW_DF_PATH):
    raw_df = pd.read_parquet(RAW_DF_PATH)
    done_ids = set(raw_df["patient_id"].unique())
else:
    raw_df = pd.DataFrame()
    done_ids = set()

pending = cohort_df[~cohort_df["patient_id"].isin(done_ids)]
print(f"[RESUME] Đã có {len(done_ids)} bệnh nhân, còn {len(pending)} cần fetch.")

MAX_RETRIES = 3
def read_full_numerics_with_retry(patient_id, record_path, timeout_sec=120):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return read_full_numerics_safe(patient_id, record_path, timeout_sec)
        except (requests.exceptions.RequestException, TimeoutError) as e:
            wait = 2 ** attempt
            print(f"[RETRY {attempt}/{MAX_RETRIES}] {patient_id} lỗi mạng ({e}), chờ {wait}s...")
            time.sleep(wait)
    print(f"[FAIL-FINAL] {patient_id} sau {MAX_RETRIES} lần thử.")
    return None

if len(pending) > 0:
    all_dfs = []
    failed_patients = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(read_full_numerics_with_retry, row["patient_id"], row["record_path"]): row["patient_id"]
                   for _, row in pending.iterrows()}
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures)):
            pid = futures[future]
            df = future.result()
            if df is not None:
                all_dfs.append(df)
            else:
                failed_patients.append(pid)

    if all_dfs:
        raw_df = pd.concat([raw_df] + all_dfs, ignore_index=True)
        raw_df.to_parquet(RAW_DF_PATH, index=False)   # ghi lại checkpoint gộp

    if failed_patients:
        with open(FAILED_LOG_PATH, "a", encoding="utf-8") as f:
            f.write("\n".join(failed_patients) + "\n")
        print(f"Vẫn còn {len(failed_patients)} bệnh nhân fail sau retry — chạy lại cell này để thử tiếp.")

### Cohort mục tiêu vs thực tế lấy được

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
counts = [len(cohort_df), raw_df["patient_id"].nunique()]
ax.bar(["Cohort mục tiêu", "raw_df thực tế"], counts, color=["#90CAF9", "#4CAF50"])
for i, v in enumerate(counts):
    ax.text(i, v, str(v), ha="center", va="bottom")
ax.set_ylabel("Số bệnh nhân")
plt.tight_layout()
plt.show()

## 1.5 Kiểm tra tổng quan `raw_df` 
Không thay đổi dữ liệu — kiểm tra fs gốc và top bệnh nhân dropout cao nhất sau resample.

In [ ]:
print(f"Số bệnh nhân trong raw_df: {raw_df['patient_id'].nunique()}")
print(f"Số dòng: {len(raw_df):,}")

fs_summary = raw_df.groupby("patient_id")["source_fs_hz"].first().round(4).value_counts()
print("\nPhân bố fs gốc (Hz) theo bệnh nhân:")
print(fs_summary)

dropout_summary = raw_df.groupby("patient_id")["dropout_ratio_full_record"].first().sort_values(ascending=False)
print("\nTop 10 bệnh nhân dropout cao nhất (sau resample 1min):")
print(dropout_summary.head(10))

## 1.6 Chọn ngưỡng dropout cho Bước 2.0
Quét thử nhiều ngưỡng `dropout_ratio_full_record` để chọn `MAX_DROPOUT_FULL` — xem biểu đồ bên dưới
để chọn điểm đánh đổi hợp lý giữa số lượng và chất lượng, thay vì chỉ đọc bảng số.

In [ ]:
patient_dropout = raw_df.groupby("patient_id")["dropout_ratio_full_record"].first()
print(patient_dropout.describe())

thresholds = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.50]
kept_counts = [(patient_dropout <= t).sum() for t in thresholds]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot([f"{t:.0%}" for t in thresholds], kept_counts, marker="o", color="#F44336")
for x, y in zip([f"{t:.0%}" for t in thresholds], kept_counts):
    ax.text(x, y, str(y), ha="center", va="bottom")
ax.set_xlabel("Ngưỡng dropout tối đa")
ax.set_ylabel("Số bệnh nhân giữ lại")
ax.set_title("Số bệnh nhân giữ lại theo ngưỡng dropout")
plt.tight_layout()
plt.show()

## 2.0 Lọc cohort: dropout ratio + độ dài bản ghi tối thiểu
Hai tiêu chí độc lập, áp dụng đồng thời:
- `dropout_ratio_full_record <= 20%` — loại bản ghi bị tháo cảm biến dài hạn.
- `duration_hours >= 6h` — loại bản ghi quá ngắn, không đủ ngữ cảnh cho rolling window 30 phút.

Ngưỡng `MAX_DROPOUT_FULL=0.25` chọn dựa trên biểu đồ ở mục 1.6.

In [ ]:
patient_stats = raw_df.groupby("patient_id").agg(
    total_rows=("SPO2", "size"),
    dropout_ratio=("dropout_ratio_full_record", "first"),
)
patient_stats["duration_hours"] = patient_stats["total_rows"] / 60

MAX_DROPOUT_FULL = 0.25
MIN_DURATION_HOURS = 5

pass_dropout = patient_stats["dropout_ratio"] <= MAX_DROPOUT_FULL
pass_duration = patient_stats["duration_hours"] >= MIN_DURATION_HOURS
patients_to_keep = patient_stats[pass_dropout & pass_duration].index

print(f"Giữ lại {len(patients_to_keep)} / {raw_df['patient_id'].nunique()} bệnh nhân")
print(f"Bị loại: {sorted(set(patient_stats.index) - set(patients_to_keep))}")

clean_df = raw_df[raw_df["patient_id"].isin(patients_to_keep)].copy()

## 2.1 Clip giá trị phi sinh lý (nhiễu cảm biến MAX30102)

In [ ]:
SPO2_MIN, SPO2_MAX = 70, 100
PULSE_MIN, PULSE_MAX = 30, 220

clean_df.loc[(clean_df["SPO2"] < SPO2_MIN) | (clean_df["SPO2"] > SPO2_MAX), "SPO2"] = np.nan
clean_df.loc[(clean_df["PULSE"] < PULSE_MIN) | (clean_df["PULSE"] > PULSE_MAX), "PULSE"] = np.nan
print(clean_df.columns.tolist())  

## 2.2 Fill gap ngắn, giữ NaN cho gap dài
Gap ngắn (≤ MAX_GAP_FILL_MIN phút liên tiếp) được forward-fill để rolling window 5-15-30 phút ở Bước 3 không bị vỡ; gap dài hơn giữ NaN, sẽ tự loại khi tính feature.

In [ ]:
MAX_GAP_FILL_MIN = 5

clean_df = clean_df.sort_values(["patient_id", "record_path", "timestamp"]).reset_index(drop=True)

clean_df[["SPO2", "PULSE"]] = (
    clean_df.groupby(["patient_id", "record_path"])[["SPO2", "PULSE"]]
    .ffill(limit=MAX_GAP_FILL_MIN)
)

post_fill_dropout = (
    clean_df.groupby("patient_id")["SPO2"]
    .apply(lambda s: s.isna().mean())
    .rename("dropout_ratio_post_fill")
)
print("Top 10 bệnh nhân còn dropout cao nhất sau fill:")
print(post_fill_dropout.sort_values(ascending=False).head(10))

## 2.3 Tổng kết Bước 2

In [ ]:
print(f"clean_df: {len(clean_df):,} dòng, {clean_df['patient_id'].nunique()} bệnh nhân")
print(f"Còn NaN sau fill: SPO2={clean_df['SPO2'].isna().sum():,}, PULSE={clean_df['PULSE'].isna().sum():,}")
clean_df.head()
clean_df.to_parquet(f"{CHECKPOINT_DIR}/clean_df.parquet", index=False)

# 3. Feature Engineering
Tính rolling stats (mean/std/min/max), slope (xu hướng), và desaturation index cho các cửa sổ 5-15-30 phút.
Mọi phép rolling đều group theo (patient_id, record_path) để không bị rò dữ liệu giữa các bệnh nhân.

## 3.0 Cấu hình & sắp xếp

In [ ]:
GROUP_KEYS = ["patient_id", "record_path"]
FEATURE_COLS = ["SPO2", "PULSE"]
WINDOWS_MIN = [5, 15, 30]

feat_df = clean_df.sort_values(GROUP_KEYS + ["timestamp"]).reset_index(drop=True)
print(feat_df.shape)

## 3.1 Rolling mean / std / min / max
`min_periods = window // 2`: yêu cầu ít nhất nửa cửa sổ có dữ liệu thật, tránh rolling toàn NaN hoặc chỉ dựa trên 1 điểm.

In [ ]:
for col in FEATURE_COLS:
    for w in WINDOWS_MIN:
        min_periods = max(1, w // 2)
        roll = feat_df.groupby(GROUP_KEYS)[col].rolling(window=w, min_periods=min_periods)
        feat_df[f"{col}_mean_{w}m"] = roll.mean().reset_index(level=GROUP_KEYS, drop=True)
        feat_df[f"{col}_std_{w}m"]  = roll.std().reset_index(level=GROUP_KEYS, drop=True)
        feat_df[f"{col}_min_{w}m"]  = roll.min().reset_index(level=GROUP_KEYS, drop=True)
        feat_df[f"{col}_max_{w}m"]  = roll.max().reset_index(level=GROUP_KEYS, drop=True)

print([c for c in feat_df.columns if "mean_5m" in c or "std_5m" in c])
feat_df[[f"SPO2_mean_5m", f"SPO2_std_5m", f"PULSE_mean_5m"]].describe()

## 3.2 Slope (xu hướng) theo từng cửa sổ
Slope = hệ số góc hồi quy tuyến tính SpO2/PULSE theo thời gian trong cửa sổ — âm mạnh nghĩa là đang xấu đi nhanh.

In [ ]:
def add_rolling_slope(feat_df, group_keys, col, window):
    """Slope tuyến tính hoá bằng công thức đóng, tương đương np.polyfit bậc 1 nhưng vector hoá hoàn toàn.
    t dùng chỉ số thời gian toàn cục trong từng (patient, record) — do slope bất biến với phép dịch t,
    kết quả giống hệt việc dùng chỉ số tương đối 0..w-1 trong hàm apply cũ."""
    min_periods = max(2, window // 2)
    t = feat_df.groupby(group_keys).cumcount().astype(float)
    y = feat_df[col]
    t_valid = t.where(y.notna()) 

    def roll_mean(s):
        return s.groupby([feat_df[group_keys[0]], feat_df[group_keys[1]]]) \
                .rolling(window=window, min_periods=min_periods).mean() \
                .reset_index(level=group_keys, drop=True)

    mean_t  = roll_mean(t_valid)
    mean_y  = roll_mean(y)
    mean_ty = roll_mean(t_valid * y)
    mean_t2 = roll_mean(t_valid ** 2)

    var_t = mean_t2 - mean_t ** 2
    cov_ty = mean_ty - mean_t * mean_y
    slope = cov_ty / var_t.replace(0, np.nan)
    return slope

for col in FEATURE_COLS:
    for w in WINDOWS_MIN:
        feat_df[f"{col}_slope_{w}m"] = add_rolling_slope(feat_df, GROUP_KEYS, col, w)

feat_df[["SPO2_slope_5m", "SPO2_slope_15m", "SPO2_slope_30m"]].describe()

## 3.3 Desaturation Index (chỉ số tụt SpO2)
Đếm số lần SpO2 tụt ≥3% so với đỉnh gần nhất trong 2 phút trước đó (1 "desaturation event"), rồi quy đổi thành số sự kiện/giờ trong mỗi cửa sổ 5-15-30 phút.

In [ ]:
DESAT_THRESHOLD = 3.0     # % SpO2 giảm được coi là 1 sự kiện
DESAT_BASELINE_MIN = 2    # phút nhìn lại để lấy baseline (đỉnh gần nhất)

feat_df["spo2_drop_from_baseline"] = (
    feat_df.groupby(GROUP_KEYS)["SPO2"]
    .transform(lambda s: s.rolling(window=DESAT_BASELINE_MIN, min_periods=1).max().shift(1) - s)
)
feat_df["is_desat_event"] = (feat_df["spo2_drop_from_baseline"] >= DESAT_THRESHOLD).astype(int)

for w in WINDOWS_MIN:
    event_count = (
        feat_df.groupby(GROUP_KEYS)["is_desat_event"]
        .rolling(window=w, min_periods=1)
        .sum()
        .reset_index(level=GROUP_KEYS, drop=True)
    )
    feat_df[f"desat_index_{w}m"] = event_count / (w / 60)  # sự kiện / giờ

feat_df[["desat_index_5m", "desat_index_15m", "desat_index_30m"]].describe()

## 3.4 Tổng kết Bước 3

In [ ]:
new_feature_cols = [c for c in feat_df.columns if c not in clean_df.columns]
print(f"Số feature mới tạo: {len(new_feature_cols)}")
print(new_feature_cols)
print(f"\nfeat_df: {feat_df.shape[0]:,} dòng x {feat_df.shape[1]} cột, {feat_df['patient_id'].nunique()} bệnh nhân")
feat_df[new_feature_cols].isna().mean().sort_values(ascending=False).head(10)
feat_df.to_parquet(f"{CHECKPOINT_DIR}/feat_df.parquet", index=False)

## Kiểm tra phân bố các feature vừa tạo
Trực quan hoá nhanh 1 vài feature đại diện (mean, slope) — phát hiện sớm nếu rolling/slope tính sai
(ví dụ slope lệch hẳn về 1 phía, hoặc mean có giá trị phi lý) thay vì chỉ đọc bảng `.describe()`.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
feat_df["SPO2_mean_15m"].hist(ax=axes[0, 0], bins=50, color="#2196F3")
axes[0, 0].set_title("SPO2_mean_15m")
feat_df["SPO2_slope_15m"].hist(ax=axes[0, 1], bins=50, color="#F44336")
axes[0, 1].set_title("SPO2_slope_15m")
feat_df["PULSE_mean_15m"].hist(ax=axes[1, 0], bins=50, color="#9C27B0")
axes[1, 0].set_title("PULSE_mean_15m")
feat_df["desat_index_15m"].hist(ax=axes[1, 1], bins=50, color="#FF9800")
axes[1, 1].set_title("desat_index_15m")
plt.tight_layout()
plt.show()

# 4. NEWS2 Labeling (rút gọn)
Sinh nhãn 4 mức (Normal/Watch/Alert/Critical) từ SpO2 + PULSE theo logic NEWS2 rút gọn.
Chỉ dùng 2/6 tham số NEWS2 gốc (thiếu RR, SBP, GCS, Temp) → điểm tối đa chỉ 0-6 thay vì 0-20.
"Critical" ở đây tương ứng "Medium risk" NEWS2 gốc, KHÔNG đạt tới "High risk" (≥7 điểm).

## 4.0 Cấu hình & hàm chấm điểm (Scale 1 / Scale 2)
Scale 1: mặc định, đa số bệnh nhân. Scale 2: dành cho bệnh nhân suy hô hấp tăng CO2 mạn (target SpO2 88-92%, VD COPD nặng) — do không có dữ liệu air/oxygen trong MIMIC numerics nên dùng bảng đối xứng quanh khoảng target, là bản rút gọn của Scale 2 gốc.

In [ ]:
NEWS2_SCALE = 1 
def score_spo2(spo2: pd.Series, scale: int = NEWS2_SCALE) -> pd.Series:
    spo2 = spo2.round()  # NEWS2 định nghĩa cho SpO2% dạng số nguyên — làm tròn trước khi tra bảng
    if scale == 1:
        conditions = [
            spo2 <= 91,
            (spo2 >= 92) & (spo2 <= 93),
            (spo2 >= 94) & (spo2 <= 95),
            spo2 >= 96,
        ]
        choices = [3, 2, 1, 0]
    elif scale == 2:
        conditions = [
            spo2 <= 83,
            (spo2 >= 84) & (spo2 <= 85),
            (spo2 >= 86) & (spo2 <= 87),
            (spo2 >= 88) & (spo2 <= 92),
            (spo2 >= 93) & (spo2 <= 94),
            (spo2 >= 95) & (spo2 <= 96),
            spo2 >= 97,
        ]
        choices = [3, 2, 1, 0, 1, 2, 3]
    else:
        raise ValueError("NEWS2_SCALE phải là 1 hoặc 2")
    return pd.Series(np.select(conditions, choices, default=np.nan), index=spo2.index)


def score_pulse(pulse: pd.Series) -> pd.Series:
    pulse = pulse.round() 
    conditions = [
        pulse <= 40,
        (pulse >= 41) & (pulse <= 50),
        (pulse >= 51) & (pulse <= 90),
        (pulse >= 91) & (pulse <= 110),
        (pulse >= 111) & (pulse <= 130),
        pulse >= 131,
    ]
    choices = [3, 1, 0, 1, 2, 3]
    return pd.Series(np.select(conditions, choices, default=np.nan), index=pulse.index)

## 4.1 Tính điểm & gán nhãn 4 mức
Ngưỡng: 0 → Normal | 1-2 → Watch | 3-4 → Alert | 5-6 → Critical.

In [ ]:
label_df = feat_df.copy()

label_df["spo2_score"] = score_spo2(label_df["SPO2"], scale=NEWS2_SCALE)
label_df["pulse_score"] = score_pulse(label_df["PULSE"])
label_df["news2_partial_score"] = label_df["spo2_score"] + label_df["pulse_score"]

def map_risk_level(score: pd.Series) -> pd.Series:
    conditions = [
        score == 0,
        (score >= 1) & (score <= 2),
        (score >= 3) & (score <= 4),
        score >= 5,
    ]
    choices = ["Normal", "Watch", "Alert", "Critical"]
    return pd.Series(np.select(conditions, choices, default="Unknown"), index=score.index)

label_df["risk_level"] = map_risk_level(label_df["news2_partial_score"])

print("Số dòng nhãn Unknown (do SPO2/PULSE còn thiếu sau Bước 2):", (label_df["risk_level"] == "Unknown").sum())

## 4.2 Kiểm tra phân bố nhãn

In [ ]:
print("Phân bố risk_level (toàn bộ):")
print(label_df["risk_level"].value_counts(dropna=False))
print("\nTỷ lệ %:")
print((label_df["risk_level"].value_counts(normalize=True, dropna=False) * 100).round(2))

patient_level_counts = label_df.groupby("patient_id")["risk_level"].value_counts().unstack(fill_value=0)
n_variety = (patient_level_counts.gt(0).sum(axis=1) > 1).sum()
print(f"\nSố bệnh nhân có >1 mức risk_level xuất hiện trong dữ liệu: {n_variety} / {label_df['patient_id'].nunique()}")
patient_level_counts.head(10)

## 4.3 Lưu checkpoint cuối

In [ ]:
label_df.to_parquet(f"{CHECKPOINT_DIR}/label_df_final.parquet", index=False)
print(f"Đã lưu {CHECKPOINT_DIR}/label_df_final.parquet — {len(label_df):,} dòng, {label_df['patient_id'].nunique()} bệnh nhân")

## 4.4 Biểu đồ phân bố nhãn

In [ ]:
import matplotlib.pyplot as plt

order = ["Normal", "Watch", "Alert", "Critical", "Unknown"]
counts = label_df["risk_level"].value_counts().reindex(order).fillna(0)

fig, ax = plt.subplots(figsize=(8, 5))
counts.plot(kind="bar", ax=ax, color=["#4CAF50", "#FFC107", "#FF9800", "#F44336", "#9E9E9E"])
ax.set_title("Phân bố risk_level (toàn bộ dòng)")
ax.set_ylabel("Số dòng")
for i, v in enumerate(counts):
    ax.text(i, v, f"{int(v):,}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

## 4.5 Sanity check nhãn NEWS2 theo 1 bệnh nhân mẫu
Chọn bệnh nhân có nhiều mức risk_level nhất để verify trực quan nhãn khớp với SpO2/PULSE thực tế.

In [ ]:
sample_patient = (
    label_df[label_df["risk_level"] != "Unknown"]
    .groupby("patient_id")["risk_level"].nunique()
    .sort_values(ascending=False)
    .index[0]
)
sample_df = label_df[label_df["patient_id"] == sample_patient].sort_values("timestamp")

fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)

axes[0].plot(sample_df["timestamp"], sample_df["SPO2"], color="#2196F3", linewidth=0.8)
axes[0].set_ylabel("SpO2 (%)")

axes[1].plot(sample_df["timestamp"], sample_df["PULSE"], color="#9C27B0", linewidth=0.8)
axes[1].set_ylabel("PULSE (bpm)")

risk_map = {"Normal": 0, "Watch": 1, "Alert": 2, "Critical": 3, "Unknown": -1}
axes[2].step(sample_df["timestamp"], sample_df["risk_level"].map(risk_map), where="post", color="#F44336")
axes[2].set_yticks(list(risk_map.values()))
axes[2].set_yticklabels(list(risk_map.keys()))
axes[2].set_ylabel("risk_level")

fig.suptitle(f"Sanity check nhãn NEWS2 — bệnh nhân {sample_patient}")
plt.tight_layout()
plt.show()

# 5. Tổng kết
*(điền sau khi chạy notebook hoàn chỉnh với cohort 850: tổng số dòng label_df, số bệnh nhân cuối cùng,
tỷ lệ % từng risk_level, số patient bị loại ở mỗi bước lọc, vấn đề còn tồn đọng nếu có.)*